In [1]:
%load_ext autoreload
%autoreload 2

auto reload en cas de modif du socle pour éviter de recharger le noyau a chaque fois

## Etape 1:

Mise en place : socle src/ml (bibliothèque du TP 14), export, train, plis, empreintes des jumeaux  
SHA-256 : c81ee13e2db77212901af38f2810e3d552b3774804bf290f2fd18582db0101e7

In [2]:
import pandas as pd

from src import config
from src.ml.evaluation import evaluate_model, fingerprint, make_cv
from src.ml.features import features_and_target, load_features, split_train_test
from src.ml.modeling import RANDOM_STATE, build_pipeline, check_no_leak
from src.storage.export_features import EXPORT_PREFIX
from src.utils import find_input

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

# Export le plus récent, nommé explicitement : il doit être celui des résultats finaux du TP 14
export_path = find_input(config.EXPORTS_DIR, EXPORT_PREFIX, ".parquet")
df = load_features(export_path)

# Seul le train est extrait : le test n'apparaîtra qu'après toutes les décisions du TP
train, _ = split_train_test(df)
X_train, y_train = features_and_target(train)
check_no_leak(build_pipeline(None), X_train)

cv = make_cv()                                # un seul objet de plis, partagé par tous les modèles
train_fingerprints = fingerprint(X_train)     # empreintes calculées une fois, pour la mesure sans jumeau

print(f"Export : {export_path.name} ({len(df):,} produits)".replace(",", " "))
print(f"Train : {len(X_train):,} produits, {X_train.shape[1]} features".replace(",", " "))

Export : features_nutriscore_20261113.parquet (459 558 produits)
Train : 367 646 produits  13 features


réentraînement des modèles du TP 14 (logistique et forêt brutes), point de contrôle 1

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

N_JOBS = -1   # coeurs pour les modèles qui le permettent ; à baisser si la mémoire sature

# Chiffres du TP 14, produits par le code du journal tp14.ipynb (F1 macro : cellule 15 ; sans jumeau : cellules 18 et 19)
TP14_REFERENCE = {
    "logistic_regression": {"f1_macro_mean": 0.7675, "f1_macro_std": 0.0019, "f1_macro_without_twin_mean": 0.7506},
    "random_forest": {"f1_macro_mean": 0.9120, "f1_macro_std": 0.0006, "f1_macro_without_twin_mean": 0.8887},
}

models = {
    "logistic_regression": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),   # multinomiale par défaut
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
}

results, oof = {}, {}
for name, model in models.items():
    results[name], oof[name] = evaluate_model(build_pipeline(model), X_train, y_train, cv, train_fingerprints)
    print(f"{name} : terminé")

# Comparaison au TP 14 : écart toléré 0,01 (point de contrôle 1)
comparison = pd.DataFrame([
    {
        "model": name,
        "tp14_f1_macro": TP14_REFERENCE[name]["f1_macro_mean"],
        "tp15_f1_macro": results[name]["f1_macro_mean"],
        "gap": results[name]["f1_macro_mean"] - TP14_REFERENCE[name]["f1_macro_mean"],
        "tp14_without_twin": TP14_REFERENCE[name]["f1_macro_without_twin_mean"],
        "tp15_without_twin": results[name]["f1_macro_without_twin_mean"],
        "gap_without_twin": results[name]["f1_macro_without_twin_mean"] - TP14_REFERENCE[name]["f1_macro_without_twin_mean"],
    }
    for name in models
]).set_index("model")
display(comparison)

max_gap = comparison[["gap", "gap_without_twin"]].abs().to_numpy().max()
print(f"Point de contrôle 1 (écart <= 0,01) : {'validé' if max_gap <= 0.01 else 'NON validé : la préparation a changé'} "
      f"(écart max {max_gap:.4f})")

logistic_regression : terminé
random_forest : terminé


,tp14_f1_macro,tp15_f1_macro,gap,tp14_without_twin,tp15_without_twin,gap_without_twin
model,,,,,,
logistic_regression,0.7675,0.7675,-1.7468e-05,0.7506,0.7506,2.3828e-05
random_forest,0.9120,0.9120,2.8751e-05,0.8887,0.8887,2.5589e-06


Point de contrôle 1 (écart <= 0,01) : validé (écart max 0.0000)


bibliothèques de boosting et de réglage disponibles, avec leur version

In [4]:
from importlib.metadata import PackageNotFoundError, version

import sklearn

# Versions citées par l'énoncé du TP 15 (environnement de référence)
EXPECTED = {"xgboost": "3.4", "lightgbm": "4.7", "catboost": "1.2", "optuna": "5"}

print(f"scikit-learn : {sklearn.__version__} (HistGradientBoostingClassifier inclus)")
for package, expected in EXPECTED.items():
    try:
        installed = version(package)
        status = "conforme" if installed.startswith(expected) else f"énoncé : {expected}.x"
        print(f"{package} : {installed} ({status})")
    except PackageNotFoundError:
        print(f"{package} : NON INSTALLÉ (énoncé : {expected}.x)")

scikit-learn : 1.9.1 (HistGradientBoostingClassifier inclus)
xgboost : 3.4.1 (conforme)
lightgbm : 4.7.0 (conforme)
catboost : NON INSTALLÉ (énoncé : 1.2.x)
optuna : 5.0.0 (conforme)


cible codée en entiers 0 à 4

In [5]:
from src.ml.features import GRADE_CODES, decode_target, encode_target

y_train_codes = encode_target(y_train)   # XGBoost exige des entiers ; les métriques sont inchangées

print(f"Codage : {GRADE_CODES}")
print(pd.DataFrame({"grade": y_train, "code": y_train_codes}).drop_duplicates().sort_values("code").to_string(index=False))

Codage : {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
grade  code
    a     0
    b     1
    c     2
    d     3
    e     4


Trois boosters avec leurs réglages par défaut (mêmes préparation, plis, métriques, cible codée)

In [6]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

boosters = {
    "hist_gradient_boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "xgboost": XGBClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
    "lightgbm": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1),
}

for name, model in boosters.items():
    results[name], oof[name] = evaluate_model(build_pipeline(model), X_train, y_train_codes, cv, train_fingerprints)
    print(f"{name} : terminé")

# Tableau provisoire : modèles du TP 14 (étape 1) et boosters par défaut
summary_columns = ["f1_macro_mean", "f1_macro_std", "f1_macro_without_twin_mean", "f1_macro_without_twin_std",
                   "fit_time_s_mean", "predict_time_s_mean"]
step2_table = pd.DataFrame(results).T[summary_columns].sort_values("f1_macro_mean")
display(step2_table)

hist_gradient_boosting : terminé
xgboost : terminé
lightgbm : terminé


,f1_macro_mean,f1_macro_std,f1_macro_without_twin_mean,f1_macro_without_twin_std,fit_time_s_mean,predict_time_s_mean
logistic_regression,0.7675,0.0019,0.7506,0.0018,23.7070,0.0718
lightgbm,0.9091,0.0013,0.8919,0.0014,22.4535,0.1866
random_forest,0.9120,0.0006,0.8887,0.0005,5.3077,0.2041
hist_gradient_boosting,0.9213,0.0016,0.9066,0.0019,7.5248,0.2146
xgboost,0.9310,0.0009,0.9170,0.0009,21.9310,0.1082


Modèle choisis : HistGradientBoosting. Le plus rapide des boosters (7,5 s par pli) et doté d’un arrêt précoce intégré, il est le seul à tenir un budget de 30 essais en un quart d’heure ; son score par défaut (0,921) est proche du meilleur, XGBoost (0,931), que ses ~22 s pour 100 arbres rendraient trop lent à régler.

#### Réglage de HistGradientBoosting — budget annoncé avant lancement

**Budget : 30 essais Optuna, 3 plis, MedianPruner, environ 15 minutes attendues (garde-fou : arrêt à 20 minutes).**

- **Critère** : F1 macro **sans jumeau**, moyenne sur 3 plis (`StratifiedKFold(3, shuffle=True, random_state=42)`),
  pour récompenser la généralisation plutôt que la mémorisation des jumeaux.
- **Pruning** : le score est rapporté après chaque pli ; un essai nettement sous la médiane des essais précédents
  est arrêté avant ses 3 plis.
- **Estimation** : ~6 s par ajustement à 100 itérations (7,5 s mesurées à 5 plis, sur 4/5 du train ; ici 2/3 du train).
  Avec l'arrêt précoce, un essai à petit pas d'apprentissage peut compter plusieurs centaines d'itérations ;
  moyenne supposée ~15 s par ajustement, soit ~45 s par essai, ~22 min sans pruning, ~15 min avec.
  Estimation incertaine : le temps réel est mesuré et noté après la recherche.

**Espace de recherche** (5 hyperparamètres) :

| Hyperparamètre | Plage | Justification |
|---|---|---|
| `learning_rate` | 0,03 – 0,3 (log) | taille de chaque correction : petit = plus d'arbres, plus lent ; plancher à 0,03 pour tenir le budget |
| `max_leaf_nodes` | 15 – 127 (log) | complexité de chaque arbre (31 par défaut) |
| `min_samples_leaf` | 10 – 200 (log) | taille minimale d'une feuille : freine la mémorisation des cas isolés |
| `l2_regularization` | 0,001 – 10 (log) | pénalité qui lisse les corrections |
| `class_weight` | aucun / `balanced` | traitement du déséquilibre, retenu pour la forêt au TP 14 |

**Fixés** : `max_iter=1200` (plafond du dépannage), arrêt précoce activé (`early_stopping=True`,
`n_iter_no_change=10`) : le nombre d'arbres est choisi par l'arrêt précoce, pas par la recherche.

Réglage de HistGradientBoosting, budget annoncé, temps réel mesuré, optimisation des réglages grace à optuna  

code commenté pour éviter la ré-éxécution, le code a remplis son role on garde l'output

In [ ]:
# import time

# import numpy as np
# import optuna
# from sklearn.base import clone
# from sklearn.model_selection import StratifiedKFold

# from src.ml.evaluation import METRICS

# N_TRIALS = 30
# TIMEOUT_S = 20 * 60   # garde-fou : la recherche s'arrête à la fin de l'essai en cours après 20 minutes
# SEARCH_CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# search_folds = list(SEARCH_CV.split(X_train, y_train_codes))   # 3 plis de recherche, calculés une fois
# f1_macro = METRICS["f1_macro"]


# def objective(trial: optuna.Trial) -> float:
#     """F1 macro sans jumeau, moyen sur les 3 plis de recherche ; arrêt anticipé si l'essai est sous la médiane."""
#     model = HistGradientBoostingClassifier(
#         learning_rate=trial.suggest_float("learning_rate", 0.03, 0.3, log=True),
#         max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 127, log=True),
#         min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 200, log=True),
#         l2_regularization=trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
#         class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
#         max_iter=1200, early_stopping=True, n_iter_no_change=10, random_state=RANDOM_STATE,
#     )
#     scores_without_twin, scores_all = [], []
#     for step, (train_idx, val_idx) in enumerate(search_folds):
#         pipeline = build_pipeline(clone(model)).fit(X_train.iloc[train_idx], y_train_codes.iloc[train_idx])
#         y_val, y_pred = y_train_codes.iloc[val_idx], pipeline.predict(X_train.iloc[val_idx])
#         has_twin = train_fingerprints.iloc[val_idx].isin(set(train_fingerprints.iloc[train_idx])).to_numpy()

#         scores_without_twin.append(f1_macro(y_val[~has_twin], y_pred[~has_twin]))
#         scores_all.append(f1_macro(y_val, y_pred))
#         trial.report(np.mean(scores_without_twin), step)
#         if trial.should_prune():
#             raise optuna.TrialPruned()

#     trial.set_user_attr("f1_macro_all", float(np.mean(scores_all)))   # chiffre sur tous les produits, pour mémoire
#     return float(np.mean(scores_without_twin))


# optuna.logging.set_verbosity(optuna.logging.WARNING)
# study = optuna.create_study(
#     direction="maximize",
#     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
#     pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
# )

# start = time.perf_counter()
# study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_S)
# search_minutes = (time.perf_counter() - start) / 60

# states = pd.Series([trial.state.name for trial in study.trials]).value_counts().to_dict()
# print(f"Temps réel : {search_minutes:.1f} min (annoncé : ~15 min) ; essais : {states}")
# print(f"Meilleur F1 macro sans jumeau (3 plis) : {study.best_value:.4f}")
# print(f"Meilleurs réglages : {study.best_params}")

# study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "duration", "state")) \
#      .sort_values("value", ascending=False).head(10)

Temps réel : 21.9 min (annoncé : ~15 min) ; essais : {'COMPLETE': 9, 'PRUNED': 4}
Meilleur F1 macro sans jumeau (3 plis) : 0.9364
Meilleurs réglages : {'learning_rate': 0.08574153197420807, 'max_leaf_nodes': 80, 'min_samples_leaf': 17, 'l2_regularization': 0.11400863701127326, 'class_weight': None}


,number,value,params_class_weight,params_l2_regularization,params_learning_rate,params_max_leaf_nodes,params_min_samples_leaf,user_attrs_f1_macro_all,duration,state
4,4,0.9364,NaN,0.1140,0.0857,80,17,0.9478,0 days 00:01:46.011109,COMPLETE
12,12,0.9364,NaN,0.0177,0.0425,83,12,0.9477,0 days 00:02:47.846002,COMPLETE
11,11,0.9360,NaN,0.0028,0.0901,66,19,0.9475,0 days 00:01:35.967490,COMPLETE
10,10,0.9357,NaN,0.0085,0.0905,52,21,0.9472,0 days 00:01:38.007421,COMPLETE
7,7,0.9357,balanced,0.4468,0.0325,105,21,0.9470,0 days 00:03:23.942730,COMPLETE
1,1,0.9351,balanced,0.6797,0.0343,95,59,0.9466,0 days 00:03:54.691197,COMPLETE
0,0,0.9351,NaN,0.2481,0.0711,115,89,0.9467,0 days 00:01:56.205867,COMPLETE
9,9,0.9344,balanced,0.0061,0.1189,108,12,NaN,0 days 00:00:38.740653,PRUNED
5,5,0.9334,NaN,6.2451,0.1215,21,12,NaN,0 days 00:00:51.197023,PRUNED
8,8,0.9327,NaN,1.2605,0.1056,22,183,NaN,0 days 00:00:50.030567,PRUNED


Budget annoncé : 30 essais, ~15 min. Réel : 22,7 min, garde-fou atteint après 13 essais (9 complets, 4 élagués). Notre train compte 367 646 produits, ~15 fois plus que l’échantillon de l’énoncé (24 076), sur lequel est calibré le repère du quart d’heure : chaque ajustement coûte d’autant plus cher. S’y ajoutent des arbres 2 à 4 fois plus grands que par défaut dans les meilleurs réglages : les essais ont duré 1,7 à 3,9 min, au lieu des ~45 s estimées. Plateau : les 7 meilleurs essais tiennent dans 0,0013 de F1 ; le réglage précis compte peu, la taille des arbres beaucoup.

meilleur réglage réévalué sur les 5 plis de mesure, gain face au défaut (point de contrôle 3)
Réglages trouvés par la recherche Optuna : la recherche n'a pas à être relancée

In [10]:
BEST_PARAMS = {
    "learning_rate": 0.08574153197420807,
    "max_leaf_nodes": 80,
    "min_samples_leaf": 17,
    "l2_regularization": 0.11400863701127326,
    "class_weight": None,
}
tuned_model = HistGradientBoostingClassifier(
    **BEST_PARAMS, max_iter=1200, early_stopping=True, n_iter_no_change=10, random_state=RANDOM_STATE
)
results["hist_gradient_boosting_tuned"], oof["hist_gradient_boosting_tuned"] = evaluate_model(
    build_pipeline(tuned_model), X_train, y_train_codes, cv, train_fingerprints
)

# Point de contrôle 3 : le gain dépasse-t-il deux fois l'écart-type des plis ?
default, tuned = results["hist_gradient_boosting"], results["hist_gradient_boosting_tuned"]
rows = []
for metric in ["f1_macro_without_twin", "f1_macro"]:   # critère de la recherche, puis chiffre du tableau
    gain = tuned[f"{metric}_mean"] - default[f"{metric}_mean"]
    fold_std = max(tuned[f"{metric}_std"], default[f"{metric}_std"])
    rows.append({"metric": metric, "default": default[f"{metric}_mean"], "tuned": tuned[f"{metric}_mean"],
                 "gain": gain, "fold_std": fold_std, "gain_over_2_std": gain > 2 * fold_std})
display(pd.DataFrame(rows).set_index("metric"))

print(f"Temps moyen d'ajustement par pli : {tuned['fit_time_s_mean']:.1f} s "
      f"(défaut : {default['fit_time_s_mean']:.1f} s)")
print(f"Pour mémoire, XGBoost par défaut : {results['xgboost']['f1_macro_mean']:.4f} "
      f"(sans jumeau : {results['xgboost']['f1_macro_without_twin_mean']:.4f})")

,default,tuned,gain,fold_std,gain_over_2_std
metric,,,,,
f1_macro_without_twin,0.9066,0.9388,0.0323,0.0019,True
f1_macro,0.9213,0.9502,0.0290,0.0016,True


Temps moyen d'ajustement par pli : 40.7 s (défaut : 7.5 s)
Pour mémoire, XGBoost par défaut : 0.9310 (sans jumeau : 0.9170)


Point de contrôle 3 : réévalué sur les 5 plis, le réglage fait passer le F1 macro de 0,921 à 0,950 (+0,029) et sans jumeau de 0,907 à 0,939 (+0,032), soit environ 8 fois deux écarts-types : gain significatif au prix d’un ajustement 5 fois plus long (41 s par pli contre 7,5 s).

Première apparition du test ; les 6 modèles ajustés sur tout le train, mesurés une seule fois

In [11]:
from src.ml.evaluation import evaluate_on_test

_, test = split_train_test(df)
X_test, y_test = features_and_target(test)
y_test_codes = encode_target(y_test)            # même codage que le train
check_no_leak(build_pipeline(None), X_test)

# Les 6 modèles du tableau, avec exactement les réglages évalués en validation croisée
final_models = {
    "logistic_regression": models["logistic_regression"],   # cellule 2
    "random_forest": models["random_forest"],               # cellule 2
    **boosters,                                             # cellule 5 : HistGradientBoosting, XGBoost, LightGBM
    "hist_gradient_boosting_tuned": tuned_model,            # cellule 8 : BEST_PARAMS
}

test_results, test_predictions, fitted = {}, {}, {}
for name, model in final_models.items():
    test_results[name], test_predictions[name], fitted[name] = evaluate_on_test(
        build_pipeline(model), X_train, y_train_codes, X_test, y_test_codes, train_fingerprints
    )
    print(f"{name} : terminé")

print(f"Test : {len(X_test):,} produits, dont {test_results['xgboost']['twin_share']:.1%} avec un jumeau dans le train"
      .replace(",", " "))

logistic_regression : terminé
random_forest : terminé
hist_gradient_boosting : terminé
xgboost : terminé
lightgbm : terminé
hist_gradient_boosting_tuned : terminé
Test : 91 912 produits  dont 21.9% avec un jumeau dans le train


Tableau unique (validation croisée et test), point de contrôle 4, quatre conditions d'égalité

In [12]:
from src.storage.export_features import SPLIT, TEST, TRAIN

rows = []
for name in final_models:
    cv_res, test_res = results[name], test_results[name]
    rows.append({
        "model": name,
        "cv_f1_macro": cv_res["f1_macro_mean"],
        "cv_f1_macro_std": cv_res["f1_macro_std"],
        "cv_f1_macro_without_twin": cv_res["f1_macro_without_twin_mean"],
        "test_f1_macro": test_res["f1_macro"],
        "test_f1_macro_without_twin": test_res["f1_macro_without_twin"],
        "test_accuracy": test_res["accuracy"],
        "gap_test_minus_cv": test_res["f1_macro"] - cv_res["f1_macro_mean"],
        "fit_time_s": test_res["fit_time_s"],          # ajustement sur tout le train
        "predict_time_s": test_res["predict_time_s"],  # prédiction des 91 912 produits du test
    })
single_table = pd.DataFrame(rows).set_index("model").sort_values("cv_f1_macro")
with pd.option_context("display.float_format", "{:.4f}".format):
    display(single_table)

# Point de contrôle 4 : écart validation croisée / test au plus 0,02 pour chaque modèle
too_far = single_table.index[single_table["gap_test_minus_cv"].abs() > 0.02].tolist()
print(f"Point de contrôle 4 (|écart| <= 0,02) : {'validé' if not too_far else f'NON validé pour {too_far}'} "
      f"(écart max {single_table['gap_test_minus_cv'].abs().max():.4f})")

# Les quatre conditions de l'égalité, vérifiées dans le code
reference_columns = fitted["xgboost"].named_steps["preprocessor"].get_feature_names_out()
conditions = {
    "même préparation : mêmes colonnes en sortie du préprocesseur pour les 6 modèles": all(
        (pipeline.named_steps["preprocessor"].get_feature_names_out() == reference_columns).all()
        for pipeline in fitted.values()
    ),
    "mêmes plis : un seul objet cv (cellule 1), identique à make_cv()": all(
        (a == b).all() for (_, a), (_, b) in zip(cv.split(X_train, y_train), make_cv().split(X_train, y_train))
    ),
    "mêmes métriques : mêmes clés de résultats pour les 6 modèles (un seul METRICS)": (
        len({frozenset(results[name]) for name in final_models}) == 1
        and len({frozenset(test_results[name]) for name in final_models}) == 1
    ),
    "même split : colonne split de l'export, train et test disjoints et complets": (
        set(train[SPLIT]) == {TRAIN} and set(test[SPLIT]) == {TEST}
        and len(train) + len(test) == len(df) and not set(train["code"]) & set(test["code"])
    ),
}
for condition, ok in conditions.items():
    print(f"{'OK ' if ok else 'ÉCHEC'} — {condition}")

,cv_f1_macro,cv_f1_macro_std,cv_f1_macro_without_twin,test_f1_macro,test_f1_macro_without_twin,test_accuracy,gap_test_minus_cv,fit_time_s,predict_time_s
model,,,,,,,,,
logistic_regression,0.7675,0.0019,0.7506,0.7691,0.7530,0.8049,0.0016,24.0251,0.0841
lightgbm,0.9091,0.0013,0.8919,0.9111,0.8930,0.9268,0.0019,45.4674,0.2196
random_forest,0.9120,0.0006,0.8887,0.9174,0.8929,0.9304,0.0054,6.6549,0.2783
hist_gradient_boosting,0.9213,0.0016,0.9066,0.9239,0.9086,0.9370,0.0026,8.5370,0.2484
xgboost,0.9310,0.0009,0.9170,0.9344,0.9203,0.9457,0.0033,43.0817,0.1272
hist_gradient_boosting_tuned,0.9502,0.0010,0.9388,0.9544,0.9427,0.9623,0.0042,57.1039,1.3100


Point de contrôle 4 (|écart| <= 0,02) : validé (écart max 0.0054)
OK  — même préparation : mêmes colonnes en sortie du préprocesseur pour les 6 modèles
OK  — mêmes plis : un seul objet cv (cellule 1), identique à make_cv()
OK  — mêmes métriques : mêmes clés de résultats pour les 6 modèles (un seul METRICS)
OK  — même split : colonne split de l'export, train et test disjoints et complets
